<a href="https://colab.research.google.com/github/acoiman/pdt/blob/main/asthma_mortality/notebooks/Python/15.Asthma_Mortality_RF_REG_SA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🥺 Sensitivity Analysis (SA) of Random Forest Regression Model to Predict  Asthma Mortality Rate in Argentina



This notebook applies Global Sensitivity Analysis (GSA) to estimate the generative importance of features within a Random Forest model for Asthma Mortality Rate (AMR) in Argentina, moving beyond predictive accuracy alone. While variable importance measures and local tools like SHAP quantify feature contributions to specific predictive outcomes, they do not clarify how variables structurally determine the response within the data-generating process [1]. Using Sobol indices, we explore the full input space to capture non-linear effects and decompose model variance into direct and interaction-driven components [1] . This analysis evaluates the structural dependence of the model on Population Density (PD), previously identified by SHAP as a primary driver. Through this generative lens, we aim to provide a more transparent and scientifically defensible explanation of the demographic and environmental factors driving asthma mortality [1] .

## 📚 Import required libraries

In [ ]:
# dataframe libraries
import pandas as pd
import numpy as np

# statistical libraries
from scipy import stats

# geospatial libraries
import geopandas as gpd
import mapclassify
from libpysal.weights import Queen
from esda.moran import Moran
from pysal.explore import esda

# plot libraries
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib.ticker import MaxNLocator
import seaborn as sns
from matplotlib.patches import Patch
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

# sklearn libraries
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error,r2_score,explained_variance_score,median_absolute_error, max_error
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import RandomizedSearchCV
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import classification_report
from sklearn.preprocessing import PowerTransformer, QuantileTransformer

# Sentivity analysis libraries
from SALib.sample import sobol as sobol_sampler
from SALib.analyze import sobol

# other libraries
import os
from joblib import Parallel, delayed
import shap
from itables import init_notebook_mode, show
from lime.lime_tabular import LimeTabularExplainer
import itertools

In [ ]:
%cd work/

In [ ]:
# Set the PROJ_LIB path
os.environ['PROJ_LIB'] = "/opt/conda/envs/gds/share/proj"

In [ ]:
 # Copy-on-Write
 pd.options.mode.copy_on_write = True

## 🆘 Helper Functions

### Functions to load and transform the dataset

In [ ]:
# path to gdf
path = "pdt/asthma_mortality/data/gpkg/data.gpkg"

In [ ]:
# Funtion to create lag variables (up to 2 years)
def create_lags(df, var, max_lag=2):
  """
  Create lagged variables (up to 2 years).

    Parameters:
    df (pd.DataFrame): Original DataFrame.
    var (str): Name of the variable to create lags for.

    Returns:
    df (pd.DataFrame): DataFrame with lagged variables.
  """

  for lag in range(1, max_lag+1):
    df[f"{var}_lag{lag}"] = df[var].shift(lag)
  return df

In [ ]:
def create_ts_long_format(path):
  """
  Create long-format time series dataset from GeoDataFrame.

    Parameters:
    path (str): Path to the GeoPackage file containing the GeoDataFrame.

    Returns:
    df_ts (pd.DataFrame): Long-format time series dataset.
  """

  # Load dataset with data per department
  gdf = gpd.read_file(path)

  # Reshape df to ts long format
  years = range(2001, 2023)
  records = []
  # iterate rows
  # exclude geometry, PD, NBUT
  for _, row in gdf.iterrows():
    iddpto = row["IDDPTO"]
    for year in years:
        records.append({
            "IDDPTO": iddpto,
            "YEAR": year,
            "AMC": row.get(f"C_{year}", np.nan), # number of Asthma Mortality C-asess
            "POP" : row.get(f"A_{year}", np.nan), # population
            "AMR": row.get(f"AMR_{year}", np.nan),
            "PM25": row.get(f"PM25_{year}", np.nan),
            "PM10": row.get(f"PM10_{year}", np.nan),
            "NBA": row.get(f"NBA_{year}", np.nan),
            "PD": row.get(f"PD_{year}", np.nan),
            "NAGRT": row.get(f"NAGRT_{year}", np.nan),
            "NNWVT": row.get(f"NNWVT_{year}", np.nan),
            "NBUT": row.get(f"NBUT_{year}", np.nan)
            })

  # create new df from list
  df_ts = pd.DataFrame(records)

  # Create lagged variables
  for var in ["NAGRT", "NNWVT", "NBUT"]:
    df_ts = create_lags(df_ts, var)

  # Drop the initial rows with NaNs due to lagging
  df_ts = df_ts.dropna().reset_index(drop=True)

  return df_ts

 ###   Funtion to Train Random Forest Regression Model on 2001–2021 Data and Predicting 2022 using Default Hyperparameters

In [ ]:
def train_eval_rfreg(df_ts, features, bin_var = "AMR_BIN",  target="AMR"):
  """
  Trains and evaluates a Random Forest Regressor model on time-series data.
  It performs a train-test split, filters for positive cases of a binary variable,
  trains the model, makes predictions, and calculates evaluation metrics.

  Parameters:
  ----------
  df_ts : pd.DataFrame
      The input DataFrame containing time-series data.
  features : list
      A list of feature column names to be used for training.
  bin_var : str, optional
      The name of the binary variable column used for filtering positive cases.
      Defaults to "AMR_BIN".
  target : str, optional
      The name of the target variable column for regression.
      Defaults to "AMR".
  Returns:
  -------
  str
      A formatted string containing R², MAE, and RMSE evaluation metrics.
  """

  # Binary target
  df_ts[bin_var] = (df_ts[target] > 0).astype(int)

  # Prepare training and test sets
  train_df = df_ts[df_ts['YEAR'] <= 2021].dropna(subset=features + [target])
  test_df = df_ts[df_ts['YEAR'] == 2022].dropna(subset=features + [target])

  # Filter for positive cases (bin_var == 1)
  train_pos = train_df[train_df[bin_var] == 1].copy()
  test_pos = test_df[test_df[bin_var] == 1].copy()

  # Define inputs
  X_train = train_pos[features]
  y_train = train_pos[target]
  X_test = test_pos[features]
  y_test = test_pos[target].reset_index(drop=True)

  # Train regression model
  reg = RandomForestRegressor(random_state=42)
  reg.fit(X_train, y_train)


  # Predict using feature columns
  y_pred = reg.predict(X_test)

  # Evaluate
  r2 = r2_score(y_test, y_pred)
  mae = mean_absolute_error(y_test, y_pred)
  rmse = mean_squared_error(y_test, y_pred, squared=False)

  return (f"R²: {r2:.3f}, MAE: {mae:.3f}, RMSE: {rmse:.3f}")

### Function for Hyperparameter Tuning of the Random Forest Regression Model

In [ ]:
def tune_rfreg_2022(
    df_ts,
    features,
    param_grid,
    target="AMR",
    bin_var="AMR_BIN",
    train_end_year=2021,
    test_year=2022,
    n_jobs=-1
):
    """
    Hyperparameter tuning for Random Forest Regression using
    training data up to train_end_year and testing on test_year.

    Parameters
    ----------
    df_ts : pd.DataFrame
        Input dataframe.

    features : list
        Predictor variables.

    param_grid : dict
        Hyperparameter grid. Example:

        {
            'n_estimators': [200, 250, 300],
            'max_depth': [10, 20, None],
            'min_samples_leaf': [1, 3, 5]
        }

    target : str, default='AMR'
        Regression target.

    bin_var : str, default='AMR_BIN'
        Binary variable identifying positive cases.

    train_end_year : int, default=2021
        Last year included in training.

    test_year : int, default=2022
        Year used for testing.

    n_jobs : int, default=-1
        Number of parallel workers.

    Returns
    -------
    best_result : dict
        Best parameter combination and metrics.

    results_df : pd.DataFrame
        Results for all parameter combinations.
    """

    # Ensure binary variable exists
    if bin_var not in df_ts.columns:
        df_ts = df_ts.copy()
        df_ts[bin_var] = (df_ts[target] > 0).astype(int)

    # Split data
    train_df = df_ts[
        (df_ts["YEAR"] <= train_end_year)
    ].dropna(subset=features + [target])

    test_df = df_ts[
        (df_ts["YEAR"] == test_year)
    ].dropna(subset=features + [target])

    # Positive cases only
    train_pos = train_df[train_df[bin_var] == 1].copy()
    test_pos = test_df[test_df[bin_var] == 1].copy()

    if train_pos.empty:
        raise ValueError("No positive training cases found.")

    if test_pos.empty:
        raise ValueError("No positive testing cases found.")

    X_train = train_pos[features].values
    y_train = train_pos[target].values

    X_test = test_pos[features].values
    y_test = test_pos[target].values

    # Generate all parameter combinations
    param_combinations = [
        dict(zip(param_grid.keys(), values))
        for values in itertools.product(*param_grid.values())
    ]

    print(f"Evaluating {len(param_combinations)} combinations")

    def evaluate_params(params):

        model = RandomForestRegressor(
            n_estimators=params["n_estimators"],
            max_depth=params["max_depth"],
            min_samples_leaf=params["min_samples_leaf"],
            n_jobs=-1,
            random_state=42
        )

        model.fit(X_train, y_train)

        y_pred = model.predict(X_test)

        mae = mean_absolute_error(y_test, y_pred)
        rmse = np.sqrt(mean_squared_error(y_test, y_pred))
        r2 = r2_score(y_test, y_pred)

        return {
            "params": params,
            "MAE": mae,
            "RMSE": rmse,
            "R2": r2
        }

    # Parallel evaluation
    results = Parallel(
        n_jobs=n_jobs,
        prefer="threads"
    )(
        delayed(evaluate_params)(params)
        for params in param_combinations
    )

    # Convert to dataframe
    results_df = pd.DataFrame(results)

    # Sort:
    # 1. lowest RMSE
    # 2. highest R²
    results_df = results_df.sort_values(
        by=["RMSE", "R2"],
        ascending=[True, False]
    ).reset_index(drop=True)

    best_result = results_df.iloc[0].to_dict()

    print("\nBest Parameters")
    print(best_result["params"])

    print("\nPerformance")
    print(f"R²   : {best_result['R2']:.3f}")
    print(f"MAE  : {best_result['MAE']:.3f}")
    print(f"RMSE : {best_result['RMSE']:.3f}")

    return best_result, results_df

### Function to Prepare Data for SOBOL Analysis

In [ ]:
def train_and_evaluate_regression_model(
    df_ts: pd.DataFrame,
    features: list,
    target: str = 'AMR',
    bin_var: str = 'AMR_BIN',
    train_end_year: int = 2021,
    test_year: int = 2022,
):
    """
    Trains a Random Forest Regressor model and evaluates its performance.

    This function prepares data, trains a RandomForestRegressor with specified
    hyperparameters, makes predictions on a test set, and calculates
    R², MAE, and RMSE for SOBOL analysis.

    Parameters:
    ----------
    df_ts : pd.DataFrame
        The input DataFrame containing time-series data, already in long format.
    features : list
        A list of feature column names to be used for training.
    target : str, optional
        The name of the target variable column for regression. Defaults to 'AMR'.
    bin_var : str, optional
        The name of the binary variable column used for filtering positive cases.
        Defaults to 'AMR_BIN'.
    train_end_year : int, optional
        The last year to include in the training set. Defaults to 2021.
    test_year : int, optional
        The year to use for the test set. Defaults to 2022.
    Returns:
    -------
    tuple
        A tuple containing:
        - reg (RandomForestRegressor): The trained Random Forest Regressor model.
        - X_train: The training feature data.
        - X_test: The test feature data.
    """

    # Ensure binary target variable exists (if not created by create_ts_long_format)
    if bin_var not in df_ts.columns:
        df_ts = df_ts.copy()
        df_ts[bin_var] = (df_ts[target] > 0).astype(int)

    # Prepare training and test sets
    train_df = df_ts[df_ts['YEAR'] <= train_end_year].dropna(subset=features + [target])
    test_df = df_ts[df_ts['YEAR'] == test_year].dropna(subset=features + [target])

    # Filter for positive cases (AMR_BIN == 1)
    train_pos = train_df[train_df[bin_var] == 1].copy()
    test_pos = test_df[test_df[bin_var] == 1].copy()

    # Define inputs
    X_train = train_pos[features]
    y_train = train_pos[target]

    X_test = test_pos[features]
    y_test = test_pos[target].reset_index(drop=True)

    # Train regression model
    reg = RandomForestRegressor(max_depth= 20,  min_samples_leaf= 3,
                                n_estimators= 200,
                                n_jobs=-1,
                                random_state=42)
    reg.fit(X_train, y_train)

    # Predict using feature columns
    y_pred = reg.predict(X_test)


    # Evaluate
    r2 = r2_score(y_test, y_pred)
    mae = mean_absolute_error(y_test, y_pred)
    rmse = mean_squared_error(y_test, y_pred, squared=False)

    print("Prediction Results:")
    print(f"R²: {r2:.3f}, MAE: {mae:.3f}, RMSE: {rmse:.3f}")

    return reg, X_train, X_test


## 🚫 Excluding Population Density (PD) as a Predictor Variable

In this section, we evaluate the robustness of the Random Forest Regression  predictions by excluding Population Density (PD). We assess how much model performance depends on this  PD predictor variable and whether predictive accuracy remains stable. This analysis examines the model’s stability and generalizability under an alternative data configuration.



In [ ]:
# apply the funtion to create long-format time series dataset from GeoDataFrame
df_ts_no_pd = create_ts_long_format(path)

In [ ]:
# visualize the dataframe
init_notebook_mode(all_interactive=True)
df_ts_no_pd

In [ ]:
# Define features excluding PD
features_no_pd = ['PM25', "PM10", 'NBA', 'NAGRT', 'NNWVT', 'NBUT',
                'NAGRT_lag1', 'NAGRT_lag2', 'NNWVT_lag1', 'NNWVT_lag2',
                'NBUT_lag1','NBUT_lag2']

In [ ]:
# train and evaluate RF regression excluding PD with dafault hiperparameters
results_rf_no_pd = train_eval_rfreg(df_ts_no_pd, features=features_no_pd)

In [ ]:
# print evaluation metrics
results_rf_no_pd

In [ ]:
# define paramter grid for hyperparameter tuning
param_grid = {
    'n_estimators': [200, 250, 300, 350],
    'max_depth': [10, 20, 30, None],
    'min_samples_leaf': [1, 3, 5]
}

In [ ]:
# apply the funtion to check RF model performamce with hyperparameter tuning
best_result_no_pd, tuning_results_no_pd = tune_rfreg_2022(
    df_ts=df_ts_no_pd,
    features=features_no_pd,
    param_grid=param_grid,
    target="AMR",
    bin_var="AMR_BIN"
)

The Random Forest regression model indicates that Population Density (PD) is a strong predictor of asthma mortality across departments. The most recent evaluation results are consistent with previous findings. Including PD yields R² ≈ 0.58; excluding PD drops R² to ≈ −0.19, below the mean-only baseline. These results suggest that PD captures spatial and demographic variance not fully accounted for by the remaining predictors. PD is therefore necessary for explaining spatial variability of asthma mortality in this model.

## 🥺  Global Sensitivity Analysis (GSA)

 The GSA  quantifies the individual contributions of predictors and their joint effects on model output [2]. Unlike local or one-at-a-time approaches, Sobol analysis explores the entire input space and explicitly accounts for nonlinear effects and interactions among predictors, making it particularly suitable for complex models such as Random Forest [3][4].

In [ ]:
# apply the funtion to create long-format time series dataset from GeoDataFrame
df_ts = create_ts_long_format(path)

In [ ]:
# Binary target
df_ts['AMR_BIN'] = (df_ts['AMR'] > 0).astype(int)

In [ ]:
# visualize the dataframe
init_notebook_mode(all_interactive=True)
df_ts

### GSA Scenario 1:  All Features Space

In [ ]:
# Define features including PD
features_pd = ['PM25', "PM10", 'PD', 'NBA', 'NAGRT', 'NNWVT', 'NBUT',
                'NAGRT_lag1', 'NAGRT_lag2', 'NNWVT_lag1', 'NNWVT_lag2',
                'NBUT_lag1','NBUT_lag2']

In [ ]:
# get trained model and data for SOBOL analysis
reg_pd, X_train_pd, X_test_pd = \
    train_and_evaluate_regression_model(
        df_ts=df_ts,
        features=features_pd,
        target='AMR',
        bin_var='AMR_BIN'
    )

In [ ]:
# Combine training and test data (if available)
all_data = pd.concat([X_train_pd, X_test_pd])

# bounds (min–max) of all data
bounds_pd = [
    [all_data[feat].min(), all_data[feat].max()]
    for feat in features_pd
]

In [ ]:
# set the Sobol problem specification
problem_pd = {
    'num_vars': len(features_pd),
    'names': features_pd,
    'bounds': bounds_pd
}

In [ ]:
# generate Sobol quasi-random samples
N = 16384  # base sample size for highly Non-Linear Models identidied by sharp
param_values_pd = sobol_sampler.sample(
    problem_pd,
    N,
    calc_second_order=True,
    seed=42
)

In [ ]:
#  convert Sobol samples into a DataFrame
X_sobol_pd = pd.DataFrame(param_values_pd, columns=features_pd)

In [ ]:
# evaluate the trained model on the Sobol-sampled input space.
# The resulting vector Y represents model output uncertainty induced by variations in the input
#Y_pd = model_output(X_sobol_pd)
Y_pd = reg_pd.predict(X_sobol_pd)

In [ ]:
# perform Sobol variance-based sensitivity analysis
sobol_results_pd = sobol.analyze(
    problem_pd,
    Y_pd,
    calc_second_order=True,
    print_to_console=False,
    seed=42
)

In [ ]:
# convert Sobol sensitivity indices into a DataFrame
sensitivity_df_pd = pd.DataFrame({
    'Feature': problem_pd['names'],
    'S1': sobol_results_pd['S1'],
    'S1_conf': sobol_results_pd['S1_conf'],
    'ST': sobol_results_pd['ST'],
    'ST_conf': sobol_results_pd['ST_conf']
}).round(2).sort_values(by='ST', ascending=False)

In [ ]:
# visualize the dataframe
init_notebook_mode(all_interactive=False)
sensitivity_df_pd

The Sobol results indicate that **NBA** is the main variance-driving predictor in the global model response. Its indices are **S1 = 0.54 (CI:0.15)** and **ST = 0.55 (CI:0.15)**, with a very small gap (**ST−S1 = 0.01**), which supports a predominantly **direct effect**. In contrast, **PD** shows **S1 = 0.16 (CI:0.09)** and **ST = 0.31 (CI:0.19)**. The larger gap for PD (**0.15**) indicates that a substantial part of its contribution is associated with **interactions** rather than only main effects. **NBUT_lag1** has smaller influence (**S1 = 0.10 CI:0.03; ST = 0.11 CI:0.03**) and is also mostly direct. Remaining variables are near zero in both S1 and ST.

Considering confidence intervals, NBA remains clearly above PD for first-order effects (approximate S1 ranges: NBA 0.39–0.69 vs PD 0.07–0.25). For total effects, separation is weaker because intervals overlap (NBA ~0.40–0.70; PD ~0.12–0.50), so PD could reach moderate total influence but still does not exceed NBA in point estimates. This suggests with high confidence that NBA is the strongest **main-effect** driver, while PD is influential with greater uncertainty and stronger interaction dependence.

Relative to SHAP (where **PD ranks first**), Sobol both **complements and refines** the interpretation. SHAP reflects contribution to individual predictions and shows PD as highly impactful at the instance level, whereas Sobol attributes global output variance primarily to NBA and secondarily to PD (especially through interactions). Thus, the two methods are not strictly contradictory: they emphasize different aspects of model behavior (local prediction influence vs global variance decomposition).

In [ ]:
# visualize second-order sensitivities (interactions)
s2_indices = []
num_vars = problem_pd['num_vars']
for i in range(num_vars):
    for j in range(i + 1, num_vars):  # Iterate only for unique pairs (i, j) where i < j
        s2_val = sobol_results_pd['S2'][i, j]
        s2_conf_val = sobol_results_pd['S2_conf'][i, j]
        feature1 = problem_pd['names'][i]
        feature2 = problem_pd['names'][j]
        s2_indices.append({
            'Feature_Pair': f"{feature1} & {feature2}",
            'S2': round(s2_val, 2),
            'S2_conf': round(s2_conf_val, 2)
        })

sensitivity_df_pd_s2 = pd.DataFrame(s2_indices).sort_values(by='S2', ascending=False)

In [ ]:
# visualize the dataframe
init_notebook_mode(all_interactive=True)
sensitivity_df_pd_s2

The table shows that Population Density (PD) has its larger second-order interactions with NBUT_lag1 (S2=0.24). The other top interaction pairs are identical: PD scores 0.23 with NAGRT, NNWVT, NBUT, and their lags. However, all of these PD interactions have confidence intervals of 0.45.

### GSA Scenario 2:  All Features Space and Population (POP) excluding PD

In [ ]:
# Define features excluding PD adding POP
features_pop = ['PM25', "PM10", 'POP', 'NBA', 'NAGRT', 'NNWVT', 'NBUT',
                'NAGRT_lag1', 'NAGRT_lag2', 'NNWVT_lag1', 'NNWVT_lag2',
                'NBUT_lag1','NBUT_lag2']

In [ ]:
# get trained model and data for SOBOL analysis
reg_pop, X_train_pop, X_test_pop = \
    train_and_evaluate_regression_model(
        df_ts=df_ts,
        features=features_pop,
        target='AMR',
        bin_var='AMR_BIN'
    )

In [ ]:
# Combine training and test data (if available)
all_data = pd.concat([X_train_pop, X_test_pop])

# bounds (min–max) of all data
bounds_pop = [
    [all_data[feat].min(), all_data[feat].max()]
    for feat in features_pop
]

In [ ]:
# set the Sobol problem specification
problem_pop = {
    'num_vars': len(features_pop),
    'names': features_pop,
    'bounds': bounds_pop
}

In [ ]:
# generate Sobol quasi-random samples
N = 16384  # base sample size for highly Non-Linear Models identidied by sharp
param_values_pop = sobol_sampler.sample(
    problem_pop,
    N,
    calc_second_order=True,
    seed=42
)

In [ ]:
#  convert Sobol samples into a DataFrame
X_sobol_pop = pd.DataFrame(param_values_pop, columns=features_pop)

In [ ]:
# evaluate the trained model on the Sobol-sampled input space.
# The resulting vector Y represents model output uncertainty induced by variations in the input
Y_pop = reg_pop.predict(X_sobol_pop)

In [ ]:
# perform Sobol variance-based sensitivity analysi
sobol_results_pop = sobol.analyze(
    problem_pop,
    Y_pop,
    calc_second_order=True,
    print_to_console=False,
    seed=42
)

In [ ]:
# convert Sobol sensitivity indices into a DataFrame
sensitivity_df_pop = pd.DataFrame({
    'Feature': problem_pop['names'],
    'S1': sobol_results_pop['S1'],
    'S1_conf': sobol_results_pop['S1_conf'],
    'ST': sobol_results_pop['ST'],
    'ST_conf': sobol_results_pop['ST_conf']
}).round(2).sort_values(by='ST', ascending=False)

In [ ]:
# visualize the dataframe
init_notebook_mode(all_interactive=True)
sensitivity_df_pop

In the absence of PD, the Sobol results indicate that **POP** is the main variance-driving predictor in the global model response. Its indices are **S1 = 0.98 (±0.30)** and **ST = 0.97 (±0.01)**, with a very small gap (**ST−S1 = 0.01**), which supports a predominantly **direct effect**.

### GSA Scenario 3:  All Features Space and Population (POP)

In [ ]:
# calculate correlation coefficient
round (df_ts['POP'].corr(df_ts['PD']),2)

In [ ]:
# Define features including PD and POP
features_pop_pd = ['PM25', "PM10", 'PD', 'POP', 'NBA', 'NAGRT', 'NNWVT', 'NBUT',
                'NAGRT_lag1', 'NAGRT_lag2', 'NNWVT_lag1', 'NNWVT_lag2',
                'NBUT_lag1','NBUT_lag2']

In [ ]:
# get trained model and data for SOBOL analysis
reg_pop_pd, X_train_pop_pd, X_test_pop_pd = \
    train_and_evaluate_regression_model(
        df_ts=df_ts,
        features=features_pop_pd,
        target='AMR',
        bin_var='AMR_BIN'
    )

In [ ]:
# Combine training and test data (if available)
all_data = pd.concat([X_train_pop_pd, X_test_pop_pd])

# bounds (min–max) of all data
bounds_pop_pd = [
    [all_data[feat].min(), all_data[feat].max()]
    for feat in features_pop_pd
]

In [ ]:
# set the Sobol problem specification
problem_pop_pd = {
    'num_vars': len(features_pop_pd),
    'names': features_pop_pd,
    'bounds': bounds_pop_pd
}

In [ ]:
# generate Sobol quasi-random samples
N = 16384  # base sample size for highly Non-Linear Models identidied by sharp
param_values_pop_pd = sobol_sampler.sample(
    problem_pop_pd,
    N,
    calc_second_order=False,
    seed=42
)

In [ ]:
#  convert Sobol samples into a DataFrame
X_sobol_pop_pd = pd.DataFrame(param_values_pop_pd, columns=features_pop_pd)

In [ ]:
# evaluate the trained model on the Sobol-sampled input space.
# The resulting vector Y represents model output uncertainty induced by variations in the input
Y_pop_pd = reg_pop_pd.predict(X_sobol_pop_pd)

In [ ]:
# perform Sobol variance-based sensitivity analysi
sobol_results_pop_pd = sobol.analyze(
    problem_pop_pd,
    Y_pop_pd,
    calc_second_order=False,
    print_to_console=False,
    seed=42
)

In [ ]:
# convert Sobol sensitivity indices into a DataFrame
sensitivity_df_pop_pd = pd.DataFrame({
    'Feature': problem_pop_pd['names'],
    'S1': sobol_results_pop_pd['S1'],
    'S1_conf': sobol_results_pop_pd['S1_conf'],
    'ST': sobol_results_pop_pd['ST'],
    'ST_conf': sobol_results_pop_pd['ST_conf']
}).round(2).sort_values(by='ST', ascending=False)

In [ ]:
# visualize the dataframe
init_notebook_mode(all_interactive=True)
sensitivity_df_pop_pd

In the presence of PD, the Sobol results indicate that **POP** is the main variance-driving predictor in the global model response. Its indices are **S1 = 0.99 (±0.33)** and **ST = 0.98 (±0.01)**, with a very small gap (**ST−S1 = 0.01**), which supports a predominantly **direct effect**.

We also observe that using PD along with Population, PD adds almost no points of additional explained variance (0.813-0.819). This means that most of the predictive information carried by PD overlaps with information already contained in Population.

## 🚩Conclussion

The application of Global Sensitivity Analysis (GSA) provides a clearer picture of the model's feature hierarchy, especially for Population Density (PD). While local interpretation (SHAP) ranked PD as the most influential feature at the instance level, the Sobol indices reveal a different global hierarchy.


* **Primary Drivers**. The analysis identifies NBA as the dominant predictor of variance in the global model response. NBA effect is predominantly direct (S1 = 0.54, ST = 0.55) (scenario 1).


* **The Role of Population Density (PD)**. PD remains a significant predictor but is secondary to NBA, and its influence is more complex. With a first-order index (S1) of 0.16 and a total index (ST) of 0.31, nearly half (Gap/ST*100) of PD's contribution comes from interactions with other variables rather than from a direct effect (scenario 1).

* **Decision on PD as a Predictor**. PD should be retained in the model. However, the analysis shows that much of its predictive signal overlaps with broader population metrics (scenario 2), and PD should not be viewed as the  dominant factor as SHAP might suggest (scenario 3).

In summary, PD is an influential component of the model's interaction structure, but NBA is the primary structural driver of the Asthma Mortality Rate in this dataset. The GSA framework reveals the generative mechanisms of the Random Forest and shifts the focus from predictive accuracy to the  generative importance of features.


##  📚 References

[1] G. Vannucci, R. Siciliano, and A. Saltelli, “Global sensitivity analysis in random forests: unveiling generative variable importance,” Stat Methods Appl, vol. 35, no.  pp. 285–317, Jun. 2026, doi: 10.1007/s10260-026-00839-y.

[2] Tosin, M., Côrtes, A.M.A., Cunha, A. (2020). A Tutorial on Sobol’ Global Sensitivity Analysis Applied to Biological Models. In: da Silva, F.A.B., Carels, N., Trindade dos Santos, M., Lopes, F.J.P. (eds) Networks in Systems Biology. Computational Biology, vol 32. Springer, Cham. https://doi.org/10.1007/978-3-030-51862-2_6

[3] Srivastava, A., Subramaniyan, A. K., &#38; Wang, L. (2017). Analytical global sensitivity analysis with Gaussian processes. <i>Ai Edam Artificial Intelligence for Engineering Design, Analysis and Manufacturing</i>, <i>31</i>(3), 235–250. https://doi.org/10.1017/S0890060417000142

[4] Ballester-Ripoll, R., Paredes, E. G., &#38; Pajarola, R. (2019). <i>Tensor Methods for Global Sensitivity Analysis</i>. 275–276. https://doi.org/10.18420/INF2019_41
